# Telecom Customer Churn- ML Classification Project
"""Alfido Tech AI-Intership Task-1
Objective: Predict whether a telecom customer will churn(leave the service) using logistic regression and Random classifiers.
Datasets: Teleco customer churn- 7043 customers, 21 features
"""

Importing Libarries

In [29]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

Loading Dataset


In [ ]:
df = pd.read_csv('data\customer_churn_prediction.ipynb')
df.head()

Viewing the Dataset

In [ ]:
df.head()
df.tail()
df.info()
df.isnull()
df.describe()
df.describe(include='object')
df.isnull().sum()
df['Churn'].value_counts()
df.shape

Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='TechSupport', hue='Churn')
plt.title('Tech support vs Churn')
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Contract vs Churn')
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='PaperlessBilling', hue='Churn')
plt.title('Paperless Billing vs Churn')
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='PaymentMethod', hue='Churn')
plt.title('PaymentMethod vs Churn')
plt.show()

In [ ]:
sns.countplot(data=df,x='InternetService',hue='Churn')
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
sns.countplot(data=df, x='OnlineSecurity', hue='Churn')
plt.title('Online Security vs Churn')
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df['tenure'], bins=20)
plt.title('Distribution of Tenure')
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges')
plt.title('Monthly Charges vs Churn')
plt.show()

Data Cleaning

In [42]:
df['TotalCharges']= pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isnull().sum()
df.dropna(inplace=True)
df.drop('customerID', axis=1, inplace=True)

Encoding

In [43]:
df['Churn']= df['Churn'].map({'Yes': 1, 'No': 0})
df=pd.get_dummies(df, drop_first=True)

Feature and Target Separation

In [44]:
X=df.drop('Churn',axis=1)
y=df['Churn']

Train-Test-Split

In [45]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Feature Scaling

In [46]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Logistic Regression

In [47]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=5000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

Model Evaluation

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_lr))

print(confusion_matrix(y_test,y_pred_lr))

print(classification_report(y_test,y_pred_lr))

Random Forest

In [49]:
#Random Forest
rf=RandomForestClassifier(random_state=42)

rf.fit(X_train,y_train)

y_pred_rf=rf.predict(X_test)

Model Evaluation

In [ ]:

print("Accuracy:",accuracy_score(y_test,y_pred_rf))

print(confusion_matrix(y_test,y_pred_rf))

print(classification_report(y_test,y_pred_rf))

Model Comparison

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy':  [accuracy_score(y_test, y_pred_lr),  accuracy_score(y_test, y_pred_rf)],
    'F1 Score':  [f1_score(y_test, y_pred_lr),        f1_score(y_test, y_pred_rf)],
    'ROC-AUC':   [
        roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]),
        roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
    ]
}).set_index('Model').round(4)

results

Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

lr_cv = cross_val_score(lr, X, y, cv=5, scoring='f1')
rf_cv = cross_val_score(rf, X, y, cv=5, scoring='f1')

print('Logistic Regression Mean F1 (CV):', lr_cv.mean().round(4))
print('Random Forest Mean F1 (CV):      ', rf_cv.mean().round(4))

Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_estimator(lr, X_test, y_test, ax=axes[0], colorbar=False)
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, ax=axes[1], colorbar=False)
axes[1].set_title('Random Forest')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

ROC plot curve

In [ ]:
ax = plt.gca()
RocCurveDisplay.from_estimator(lr, X_test, y_test, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_estimator(rf, X_test, y_test, name='Random Forest', ax=ax)
plt.title('ROC Curves — Model Comparison')
plt.plot([0,1], [0,1], 'k--', label='Random baseline')
plt.legend()
plt.show()

Feature Importance Plot

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, palette='Blues_r')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

Saving Model

In [56]:
import joblib

joblib.dump(lr, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

CONCLUSION
## Conclusion

### Dataset
Telco Customer Churn dataset — 7043 customers, 21 features.  
Goal: predict whether a telecom customer will leave the service (binary classification).  
After cleaning, 7032 rows were used (11 rows dropped due to blank TotalCharges values).

### Preprocessing Steps
- Converted `TotalCharges` from text to numeric using `pd.to_numeric()`, dropped 11 null rows
- Dropped `customerID` — not useful for prediction
- Encoded `Churn` column: Yes → 1, No → 0
- One-hot encoded all categorical columns using `pd.get_dummies(drop_first=True)`
- Applied `StandardScaler` to normalize all features before training

### Model Comparison Results

| Metric | Logistic Regression | Random Forest |
|--------|-------------------|---------------|
| Accuracy | 78.75% | 78.46% |
| Precision | 62.06% | 62.54% |
| Recall | 51.60% | 47.33% |
| F1 Score | 56.35% | 53.88% |
| ROC-AUC | 0.8319 | 0.8153 |
| CV Mean F1 (5-fold) | 0.5994 | 0.5485 |

### Confusion Matrix Summary

**Logistic Regression:**
- Correctly identified 193 churners (True Positives)
- Missed 181 churners (False Negatives)
- Falsely flagged 118 non-churners (False Positives)

**Random Forest:**
- Correctly identified 177 churners (True Positives)
- Missed 197 churners (False Negatives)
- Falsely flagged 106 non-churners (False Positives)

### Winner: Logistic Regression
Despite being the simpler model, Logistic Regression outperformed Random Forest on 
this dataset across F1 Score (0.5635 vs 0.5388), ROC-AUC (0.8319 vs 0.8153), 
and Cross-Validation Mean F1 (0.5994 vs 0.5485). It also catches more actual 
churners (Recall: 51.6% vs 47.3%), which is the most important metric for a 
churn use case — missing a churner costs more than a false alarm.

### Key Insights from EDA
- **Contract type** is the strongest predictor — month-to-month customers churn far more than yearly contract customers
- **Electronic check** payment method correlates with significantly higher churn rates
- **Customers without tech support** are more likely to leave
- **Short tenure** customers churn more — loyalty naturally increases with time
- **Paperless billing** customers show slightly higher churn
- **Gender** shows no meaningful difference in churn rate — confirmed by model feature importance

### Feature Importance (Random Forest — Top Factors)
tenure, MonthlyCharges, and TotalCharges were the top numeric predictors.
Among categorical features, Contract type and PaymentMethod had the highest importance scores,
which directly validates the patterns observed in EDA plots.

### What I Would Try Next
- Handle class imbalance (~73% No vs ~27% Yes) using **SMOTE** oversampling
- Hyperparameter tuning with **GridSearchCV** on both models
- Try **XGBoost** or **Gradient Boosting** for potentially higher performance
- Apply **SHAP values** for deeper explainability of individual predictions (Project 4)